<a href="https://colab.research.google.com/github/cesarpc1307/Quorum-IA-Detector/blob/main/Qu%C3%B3rum_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Preparación de librerías
!pip install python-docx transformers torch -q

import docx
import os
import time # <--- Necesario para la estabilidad
from google.colab import files
from transformers import pipeline
from IPython.display import clear_output

# 2. Inicialización de los 3 Jueces (GPU T4)
print("Iniciando Escuadrón de Jueces en la GPU T4...")
juez_1 = pipeline("text-classification", model="roberta-base-openai-detector", device=0)
juez_2 = pipeline("text-classification", model="Hello-SimpleAI/chatgpt-detector-roberta", device=0)
juez_3 = pipeline("text-classification", model="openai-community/roberta-large-openai-detector", device=0)

print("✅ Los 3 jueces están listos para la deliberación.")
time.sleep(1) # Pausa breve para confirmar carga

def auditoria_quorum_ia(ruta):
    if isinstance(ruta, list): ruta = ruta
    ruta = str(ruta).replace("[", "").replace("]", "").replace("'", "")

    if not os.path.exists(ruta):
        return print(f"❌ Error: El archivo '{ruta}' no fue encontrado.")

    doc = docx.Document(ruta)
    parrafos = [p.text.strip() for p in doc.paragraphs if len(p.text.strip()) > 40]

    print(f"\n--- ANALIZANDO DOCUMENTO: {ruta} ---")
    print(f"Bloques de texto detectados: {len(parrafos)}\n")

    ia_j1, ia_j2, ia_j3, conteo_consenso = 0, 0, 0, 0

    try:
        res_j1 = juez_1(parrafos, truncation=True, max_length=512)
        res_j2 = juez_2(parrafos, truncation=True, max_length=512)
        res_j3 = juez_3(parrafos, truncation=True, max_length=512)

        for i in range(len(parrafos)):
            def limpiar_res(r):
                while isinstance(r, list): r = r
                return r

            d1, d2, d3 = limpiar_res(res_j1[i]), limpiar_res(res_j2[i]), limpiar_res(res_j3[i])
            es_ia_1 = (d1['label'] == 'Fake')
            es_ia_2 = (d2['label'].upper() in ['CHATGPT', 'LABEL_1', 'FAKE'])
            es_ia_3 = (d3['label'] == 'Fake')

            if es_ia_1: ia_j1 += 1
            if es_ia_2: ia_j2 += 1
            if es_ia_3: ia_j3 += 1

            votos_ia = int(es_ia_1) + int(es_ia_2) + int(es_ia_3)
            if votos_ia >= 2: marca, peso = "🔴 [IA - MAYORÍA]", 1
            elif votos_ia == 1: marca, peso = "🟡 [DUDA / REVISAR]", 0.5
            else: marca, peso = "🟢 [HUMANO]", 0

            conteo_consenso += peso
            print(f"P{i+1:02d}: {marca} (Votos IA: {votos_ia}/3) | {parrafos[i][:60]}...", flush=True)

        total = len(parrafos)
        for j, (nombre, hits, desc) in enumerate([
            ("RoBERTa Base", ia_j1, "Analiza patrones estadísticos y predictibilidad."),
            ("ChatGPT Detector", ia_j2, "Especializado en la huella digital de modelos GPT."),
            ("RoBERTa Large", ia_j3, "Evaluación de contexto profundo (355M params).")
        ]):
            print("\n" + "-"*45)
            print(f"⚖️ RESUMEN JUEZ {j+1} ({nombre})")
            print(f"Descripción: {desc}")
            print(f"RESULTADO -> IA: {hits} | Humano: {total-hits} | Prob: {(hits/total)*100:.2f}%")

        pct_final = (conteo_consenso / total) * 100
        print("\n" + "="*45)
        print(f"📊 INFORME DE CONSENSO TÉCNICO (PROMEDIO)")
        print(f"="*45)
        print(f"Párrafos procesados: {total}")
        print(f"Índice de Probabilidad IA: {pct_final:.2f}%")
        print(f"Veredicto: {'ALTA SOSPECHA' if pct_final > 20 else 'INTEGRIDAD VERIFICADA'}")

        print("\n" + "—"*45)
        print(f"🛠️ Auditoría técnica realizada por: Ing. César Pineda")
        print(f"🇭🇳 Gracias, Lempira, Honduras")
        print(f"📌 Proyecto: Quórum-IA - Verificación de Integridad")
        print("—"*45)

    except Exception as e:
        print(f"❌ Error en el proceso: {e}")

# 3. GESTOR DE ARCHIVOS (Con corrección de limpieza de pantalla)
def iniciar_sistema():
    # Primero limpiamos todo lo que soltó Hugging Face
    clear_output()
    time.sleep(0.5) # Pausa técnica para asegurar que el input se dibuje bien

    archivos_locales = [f for f in os.listdir('.') if f.endswith('.docx')]

    print("📂 --- GESTOR DE ARCHIVOS QUÓRUM-IA ---")
    print(" ARCHIVOS EXISTENTES ")

    for idx, nombre in enumerate(archivos_locales):
        print(f"[{idx + 1}] 📄 Usar archivo existente: {nombre}")

    print("\n👉 Selecciona una opción (0 para subir):")
    try:
        entrada = input("> ")
        if not entrada: return # Evita error si presionas enter vacío

        seleccion = int(entrada)

        if seleccion == 0:
            print("🚀 Seleccionando archivo...")
            uploaded = files.upload()
            if uploaded:
                nombre_subido = list(uploaded.keys())
                auditoria_quorum_ia(nombre_subido)
        elif 1 <= seleccion <= len(archivos_locales):
            archivo_elegido = archivos_locales[seleccion - 1]
            auditoria_quorum_ia(archivo_elegido)
        else:
            print("❌ Opción no válida.")
    except ValueError:
        print("❌ Error: Introduce un número.")

# EJECUTAR
iniciar_sistema()